<a href="https://colab.research.google.com/github/daureny/Dashboard_PyCh_final/blob/master/New_Banking_Dash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dash dash-bootstrap-components plotly pandas flask-caching gunicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 14.4 MB/s eta 0:00:00
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.0
    Uninstalling Flask-3.1.0:
      Successfully uninstalled Flask-3.1.0


In [3]:
# Import necessary libraries
from dash import Dash, dcc, html, Input, Output, callback_context
import pandas as pd
import plotly.graph_objs as go
import dash_bootstrap_components as dbc
from flask_caching import Cache
import traceback
from functools import lru_cache
import gunicorn

# Constants
GIT_PATH = 'https://github.com/daureny/Dashboard_PyCh_final/raw/master/Data'

# Initialize the app
app = Dash(__name__,
           external_stylesheets=[dbc.themes.YETI],
           meta_tags=[{'name': 'viewport',
                       'content': 'width=device-width, initial-scale=1.0'}]
           )
server = app.server

# Setup caching for data loading
cache = Cache(app.server, config={
    'CACHE_TYPE': 'filesystem',
    'CACHE_DIR': 'cache-directory',
    'CACHE_DEFAULT_TIMEOUT': 3600  # 1 hour cache timeout
})

# Data loading functions with caching
@cache.memoize()
def load_financial_indicators():
    """
    Load financial indicators data from Excel file.

    Returns:
        pandas.DataFrame: Processed financial indicators data
    """
    try:
        workbook = pd.ExcelFile(f'{GIT_PATH}/FI2.xlsx')
        sheets = workbook.sheet_names

        # Concatenate all sheets and add date column
        df = pd.concat([
            pd.read_excel(workbook, sheet_name=s).assign(Дата=s)
            for s in sheets
        ])

        # Process dataframe
        df['Наименование банка'] = df['Наименование банка'].astype('category')
        df = df.set_index('Наименование банка')

        # Select only needed columns directly instead of dropping
        keep_cols = [col for col in df.columns[:19] if col != '№']
        df = df[keep_cols]

        df = df.fillna(0)
        return df
    except Exception as e:
        print(f"Error loading financial indicators: {str(e)}")
        # Return empty dataframe with expected columns to avoid breaking app
        return pd.DataFrame(columns=['Дата', 'Активы', 'Ссудный портфель',
                                    'Просрочка свыше 90 дней', 'Провизии по МСФО'])

@cache.memoize()
def load_loan_portfolio():
    """
    Load loan portfolio data from Excel file.

    Returns:
        pandas.DataFrame: Processed loan portfolio data
    """
    try:
        df_LP_raw = pd.read_excel(io=f'{GIT_PATH}/LP.xlsx', engine='openpyxl')
        df = df_LP_raw.transpose()

        df.columns = df.iloc[0]
        df = df.drop('Наименование показателя')
        df.index.name = 'Дата'
        df.columns.name = ''

        # Ensure dates are properly formatted
        df.index = pd.to_datetime(df.index, format='%Y-%m-%d', errors='coerce')
        df = df.sort_index(ascending=True)

        return df
    except Exception as e:
        print(f"Error loading loan portfolio: {str(e)}")
        return pd.DataFrame()

@cache.memoize()
def load_interest_margin():
    """
    Load interest margin data from Excel file.

    Returns:
        pandas.DataFrame: Processed interest margin data
    """
    try:
        workbook = pd.ExcelFile(f'{GIT_PATH}/IM.xlsx')
        sheets = workbook.sheet_names

        df = pd.concat([
            pd.read_excel(workbook, sheet_name=s).assign(Дата=s)
            for s in sheets
        ])

        # Process dataframe
        columns_to_keep = df.columns[:11]
        columns_to_keep = [col for col in columns_to_keep if col != 'N            п/п']
        df = df[columns_to_keep]

        df = df.drop(0)
        df['Наименование банка'] = df['Наименование банка'].astype('category')
        df = df.set_index('Наименование банка')
        df = df.drop('2')

        # Scale and rename columns
        df['Процентная маржа'] = df['Процентная маржа'] * 100
        df = df.rename(columns={
            "Активы, приносящие доход (нетто)1": "Активы, приносящие доход (нетто)",
            "Активы, приносящие доход (брутто)1": "Активы, приносящие доход (брутто)",
            'Обязательства, связанные с выплатой вознаграждения1': 'Обязательства, связанные с выплатой вознаграждения',
            'Доходы, связанные с получением вознаграждения2': 'Доходы, связанные с получением вознаграждения',
            'Расходы, связанные с выплатой вознаграждения2': 'Расходы, связанные с выплатой вознаграждения'
        })

        return df
    except Exception as e:
        print(f"Error loading interest margin: {str(e)}")
        return pd.DataFrame()

@cache.memoize()
def load_prudential_norms():
    """
    Load prudential norms data from Excel file.

    Returns:
        tuple: (df_PN, df_PNT, df_floor_threshold, coefs)
            - df_PN: Prudential norms dataframe
            - df_PNT: Threshold values dataframe
            - df_floor_threshold: Floor threshold dataframe
            - coefs: List of coefficient names
    """
    try:
        # Load prudential norms
        workbook = pd.ExcelFile(f'{GIT_PATH}/PN.xlsx')
        sheets = workbook.sheet_names

        df_PN = pd.concat([
            pd.read_excel(workbook, sheet_name=s).assign(Дата=s)
            for s in sheets
        ])

        # Select and process columns
        columns_to_keep = [col for col in df_PN.columns[:31] if col not in ['№ п/п', 'Собственный капитал ']]
        df_PN = df_PN[columns_to_keep]
        df_PN = df_PN.set_index('Наименование банков второго уровня')

        # Load thresholds
        df_PNT = pd.read_excel(io=f'{GIT_PATH}/PN_threshold.xlsx', engine='openpyxl')
        df_PNT = df_PNT.set_index(df_PNT['Unnamed: 0'])
        df_PNT.drop(df_PNT.columns[[0, 2, 3]], axis=1, inplace=True)
        df_PNT = df_PNT.rename(columns={'Unnamed: 1': 'T'})

        # Select coefficients with floor threshold
        df_floor_threshold = pd.concat([
            df_PNT.iloc[0:3],     # Rows from index 0 to 2
            df_PNT.iloc[8:18],    # Rows from index 8 to 17
            df_PNT.iloc[19:22]    # Rows from index 19 to 21
        ])

        # Get list of coefficients
        coefs = [name for name in df_PN.columns if name != 'Дата']

        return df_PN, df_PNT, df_floor_threshold, coefs
    except Exception as e:
        print(f"Error loading prudential norms: {str(e)}")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), []

# Load all datasets
try:
    df_FI = load_financial_indicators()
    df_LP = load_loan_portfolio()
    df_IM = load_interest_margin()
    df_PN, df_PNT, df_floor_threshold, coefs = load_prudential_norms()
except Exception as e:
    print(f"Error during initial data loading: {str(e)}")

# Helper functions for creating graphs
def create_line_chart(df, x_col, y_col, title, category_col=None):
    """
    Create a line chart for the given data.

    Args:
        df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        title: Chart title
        category_col: Column name for categories (default: None)

    Returns:
        plotly.graph_objs.Figure: Line chart figure
    """
    data = []

    if category_col:
        for category in df[category_col].unique():
            df_filtered = df[df[category_col] == category]
            trace = go.Scatter(
                x=df_filtered[x_col],
                y=df_filtered[y_col],
                mode='markers+lines',
                name=category,
                line=dict(width=3)
            )
            data.append(trace)
    else:
        trace = go.Scatter(
            x=df[x_col],
            y=df[y_col],
            mode='markers+lines',
            line=dict(width=3)
        )
        data.append(trace)

    layout = go.Layout(
        title={
            'text': title,
            'y': 0.9,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 18, 'color': 'black', 'family': "Arial Black, sans-serif"}
        },
        template='plotly_white',  # Use a cleaner template
        margin=dict(l=50, r=50, t=80, b=50),  # Adjust margins
    )

    return go.Figure(data=data, layout=layout)

def create_pie_chart(values, labels, title):
    """
    Create a pie chart for the given data.

    Args:
        values: List of values
        labels: List of labels
        title: Chart title

    Returns:
        plotly.graph_objs.Figure: Pie chart figure
    """
    data = go.Pie(
        labels=labels,
        values=values,
        textinfo='percent+label',
        insidetextorientation='radial'
    )

    layout = go.Layout(
        title={
            'text': title,
            'y': 0.9,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 18, 'color': 'black', 'family': "Arial Black, sans-serif"}
        },
        margin=dict(l=20, r=20, t=80, b=20),
    )

    return go.Figure(data=data, layout=layout)

def create_stacked_bar_chart(df, x_col, y_cols, title, height=600, width=800):
    """
    Create a stacked bar chart for the given data.

    Args:
        df: DataFrame containing the data
        x_col: Column name or index for x-axis
        y_cols: List of column names for y-axis (stacks)
        title: Chart title
        height: Chart height
        width: Chart width

    Returns:
        plotly.graph_objs.Figure: Stacked bar chart figure
    """
    # Create traces for each y column
    data = [
        go.Bar(
            y=df[col],
            x=df.index if x_col is None else df[x_col],
            name=col
        ) for col in y_cols
    ]

    layout = go.Layout(
        title={
            'text': title,
            'y': 0.9,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 18, 'color': 'black', 'family': "Arial Black, sans-serif"}
        },
        barmode='stack',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,
            xanchor="center",
            x=0.5
        ),
        height=height,
        width=width,
        margin=dict(l=50, r=50, t=80, b=150),  # Extra bottom margin for legend
    )

    return go.Figure(data=data, layout=layout)

# App Layout with improved organization and styling
app.layout = dbc.Container([
    # Header section
    dbc.Row([
        dbc.Col([
            html.H2('Основные показатели банков',
                   className='text-center mt-4 mb-4',
                   style={'fontWeight': 'bold'}),
            html.P('На данном интерактивном дешборде вы можете просмотреть статистику по банкам. '
                  'Кликайте на график для того, чтобы отфильтровать те или иные банки. '
                  'Источником данных является сайт Национального Банка РК (www.nationalbank.kz)',
                  className='lead text-center mb-4')
        ], width=12)
    ]),

    # Loading indicator for the entire dashboard
    dbc.Spinner(
        children=[
            # Graph 1 - Bank indicators dynamics
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='graph-1', config={'responsive': True}),
                    dbc.Row([
                        dbc.Col([
                            dcc.Dropdown(
                                id='line-y',
                                options=[
                                    {'label': 'Активы', 'value': 'Активы'},
                                    {'label': 'Ссудный портфель', 'value': 'Ссудный портфель'},
                                    {'label': 'Просрочка свыше 90 дней', 'value': 'Просрочка свыше 90 дней'},
                                    {'label': 'Провизии по МСФО', 'value': 'Провизии по МСФО'},
                                    {'label': 'Обязательства', 'value': 'Обязательства'},
                                    {'label': 'Собственный капитал по балансу', 'value': 'Собственный капитал по балансу'},
                                    {'label': 'Превышение текущих доходов (расходов) над текущими '
                                            'расходами (доходами) после уплаты подоходного налога',
                                    'value': 'Превышение текущих доходов (расходов) над текущими '
                                            'расходами (доходами) после уплаты подоходного налога'}
                                ],
                                value='Активы',
                                clearable=False,
                                className='mb-4'
                            )
                        ], width={'size': 8, 'offset': 2})
                    ])
                ], width=12)
            ]),

            # Graphs 2 & 3 - Asset quality
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='graph-2', config={'responsive': True})
                ], xs=12, md=6),
                dbc.Col([
                    dcc.Graph(id='graph-3', config={'responsive': True})
                ], xs=12, md=6)
            ]),

            # Selectors for Graphs 2 & 3
            dbc.Row([
                dbc.Col([
                    dcc.Dropdown(
                        id='bank_name',
                        options=[{'label': bank, 'value': bank} for bank in df_FI.index.unique()],
                        value=df_FI.index.unique()[0] if not df_FI.empty else None,
                        clearable=False,
                        className='mb-4'
                    )
                ], xs=12, md=6, lg={'size': 6, 'offset': 2}),
                dbc.Col([
                    dcc.Dropdown(
                        id='date',
                        options=[{'label': date, 'value': date} for date in df_FI['Дата'].unique()],
                        value=df_FI['Дата'].min() if not df_FI.empty else None,
                        clearable=False,
                        className='mb-4'
                    )
                ], xs=12, md=6, lg=4)
            ]),

            # Graphs 4 & 5 - Loan portfolio
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='graph-4', config={'responsive': True})
                ], xs=12, md=6),
                dbc.Col([
                    dcc.Graph(id='graph-5', config={'responsive': True})
                ], xs=12, md=6)
            ]),

            # Date selectors for Graphs 4 & 5
            dbc.Row([
                dbc.Col([
                    html.Label('Start Date', className='font-weight-bold'),
                    dcc.Dropdown(
                        id='date_start',
                        options=[{'label': date.strftime('%Y-%m-%d'), 'value': date}
                                for date in df_LP.index.unique()],
                        value=df_LP.index.min() if not df_LP.empty else None,
                        clearable=False,
                        className='mb-4'
                    )
                ], xs=12, md=6, lg={'size': 4, 'offset': 2}),
                dbc.Col([
                    html.Label('End Date', className='font-weight-bold'),
                    dcc.Dropdown(
                        id='date_end',
                        options=[{'label': date.strftime('%Y-%m-%d'), 'value': date}
                                for date in df_LP.index.unique()],
                        value=df_LP.index.max() if not df_LP.empty else None,
                        clearable=False,
                        className='mb-4'
                    )
                ], xs=12, md=6, lg=4)
            ]),

            # Graph 6 - Interest margin
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='graph-6', config={'responsive': True}),
                    dbc.Row([
                        dbc.Col([
                            dcc.Dropdown(
                                id='dd_graph-6',
                                options=[{'label': name, 'value': name} for name in df_IM.columns if name != 'Дата'],
                                value='Процентная маржа' if 'Процентная маржа' in df_IM.columns else None,
                                clearable=False,
                                className='mb-4'
                            )
                        ], width={'size': 10, 'offset': 1})
                    ])
                ], width=12)
            ]),

            # Graph 7 - Prudential norms
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='graph-7', config={'responsive': True}),
                    dbc.Row([
                        dbc.Col([
                            dcc.Dropdown(
                                id='dd_graph-7',
                                options=[{'label': name, 'value': name} for name in coefs],
                                value=coefs[0] if coefs else None,
                                clearable=False,
                                className='mb-4'
                            )
                        ], width={'size': 10, 'offset': 1})
                    ])
                ], width=12)
            ])
        ],
        color="primary",
        type="border",
        fullscreen=True,
    ),

    # Footer section
    dbc.Row([
        dbc.Col([
            html.Hr(),
            html.P('ПРЕДУПРЕЖДЕНИЕ: полнота, достоверность и точность данных, представленных данным дешбордом зависит от соответствующих данных, '
                 'опубликованных на сайте Национального Банка РК (www.nationalbank.kz). ТОО "Стандарт бизнес консалтинг" не несет ответственности '
                 'за полноту, точность и достоверность представленных данных',
                 className='text-muted small'),
            html.P('ТОО "Стандарт бизнес консалтинг", 2023. Все права защищены.',
                  className='text-muted text-center small')
        ], width=12)
    ])
], fluid=True, className='px-4')  # Add padding for better mobile view

# Callback functions with improved error handling
@app.callback(
    Output("graph-1", "figure"),
    Input("line-y", "value")
)
def update_bank_indicators_chart(selected_item):
    """Update the bank indicators chart based on the selected item."""
    try:
        # Create traces
        data = []

        for bank in df_FI.index.unique():
            bank_data = df_FI[df_FI.index == bank]

            # Skip if no data for this bank
            if bank_data.empty:
                continue

            trace = go.Scatter(
                x=bank_data['Дата'],
                y=bank_data[selected_item],
                mode='markers+lines',
                name=bank,
                line=dict(width=3)
            )
            data.append(trace)

        layout = go.Layout(
            title={
                'text': '1. Динамика показателей банков по статьям финансовой отчетности',
                'y': 0.9,
                'x': 0.5,
                'xanchor': 'center',
                'yanchor': 'top',
                'font': {'size': 18, 'color': 'black', 'family': "Arial Black, sans-serif"}
            },
            xaxis_title="Дата",
            yaxis_title=selected_item,
            template='plotly_white',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=-0.3,
                xanchor="center",
                x=0.5
            ),
            margin=dict(l=50, r=50, t=80, b=150),  # Extra bottom margin for legend
        )

        return go.Figure(data=data, layout=layout)
    except Exception as e:
        print(f"Error updating graph-1: {str(e)}")
        # Return empty figure with error message
        return go.Figure().add_annotation(
            text=f"Error loading data: {str(e)}",
            showarrow=False,
            font=dict(size=14, color="red")
        )

@app.callback(
    [Output("graph-2", "figure"),
     Output("graph-3", "figure")],
    [Input("bank_name", "value"),
     Input("date", "value")]
)
def update_asset_quality_charts(bank_name, date):
    """Update the asset quality charts based on selected bank and date."""
    try:
        # Check if data is available
        if df_FI.empty or bank_name is None or date is None:
            raise ValueError("Data, bank name, or date not available")

        # Graph 2 - Overdue loans
        delinquency_labels = ['Ссудный портфель', 'Просрочка свыше 7 дней', 'Просрочка свыше 90 дней']
        delinquency_values = []

        # Graph 3 - Provisions
        provision_labels = ['Ссудный портфель', 'Провизии по МСФО']
        provision_values = []

        # Get data for selected bank and date
        bank_data = df_FI[(df_FI.index == bank_name) & (df_FI['Дата'] == date)]

        if bank_data.empty:
            raise ValueError(f"No data for bank {bank_name} on date {date}")

        # Extract values for Graph 2
        for label in delinquency_labels:
            if label in bank_data.columns:
                delinquency_values.append(float(bank_data[label].iloc[0]))
            else:
                delinquency_values.append(0)

        # Extract values for Graph 3
        for label in provision_labels:
            if label in bank_data.columns:
                provision_values.append(float(bank_data[label].iloc[0]))
            else:
                provision_values.append(0)

        # Create figures
        fig2 = create_pie_chart(
            values=delinquency_values,
            labels=delinquency_labels,
            title='2. Качество активов - просрочки'
        )

        fig3 = create_pie_chart(
            values=provision_values,
            labels=provision_labels,
            title='3. Качество активов - провизии'
        )

        return fig2, fig3
    except Exception as e:
        print(f"Error updating asset quality charts: {str(e)}")
        # Return empty figures with error message
        empty_fig = go.Figure().add_annotation(
            text=f"Error loading data: {str(e)}",
            showarrow=False,
            font=dict(size=14, color="red")
        )
        return empty_fig, empty_fig

@app.callback(
    [Output("graph-4", "figure"),
     Output("graph-5", "figure")],
    [Input("date_start", "value"),
     Input("date_end", "value")]
)
def update_loan_portfolio_charts(date_start, date_end):
    """Update the loan portfolio charts based on selected date range."""
    try:
        # Check if data is available
        if df_LP.empty or date_start is None or date_end is None:
            raise ValueError("Data or date range not available")

        # Convert dates to datetime if they aren't already
        date_start = pd.to_datetime(date_start)
        date_end = pd.to_datetime(date_end)

        # Make sure end date is not before start date
        if date_end < date_start:
            date_start, date_end = date_end, date_start

        # Graph 4 - Loan portfolio by type
        loan_type_labels = [
            'Межбанковские займы',
            'Операции «Обратное РЕПО»',
            'Займы небанковским юридическим лицам и индивидуальным предпринимателям (включая нерезидентов), '
            'за исключением субъектов малого и среднего предпринимательства – резидентов РК',
            'Займы небанковским юридическим лицам и индивидуальным предпринимателям - резидентам РК, являющимся '
            'субъектами малого и среднего предпринимательства',
            'Займы физическим лицам (включая нерезидентов), за исключением кредитов индивидуальным предпринимателям на '
            'предпринимательские цели'
        ]

        # Filter columns that exist in the dataframe
        loan_type_labels = [col for col in loan_type_labels if col in df_LP.columns]
        df_LP_type = df_LP[loan_type_labels].loc[date_start:date_end]

        # Graph 5 - Loan portfolio provisions
        provision_labels = [
            'Займы, по которым отсутствует просроченная задолженность по основному долгу и/или начисленному '
            'вознаграждению ',
            'Займы с просроченной задолженностью от 1 до 30 дней',
            'Займы с просроченной задолженностью от 31 до 60 дней',
            'Займы с просроченной задолженностью от 61 до 90 дней',
            'Займы с просроченной задолженностью свыше 90 дней',
            'Провизии по МСФО',
            'Провизии по займам с просроченной задолженностью свыше 90 дней'
        ]

        # Filter columns that exist in the dataframe
        provision_labels = [col for col in provision_labels if col in df_LP.columns]
        df_LP_provision = df_LP[provision_labels].loc[date_start:date_end]

        # Create figures
        fig4 = create_stacked_bar_chart(
            df=df_LP_type,
            x_col=None,  # Use index as x-axis
            y_cols=loan_type_labels,
            title='4. Ссудный портфель банков в разрезе видов',
            height=600,
            width=800
        )

        fig5 = create_stacked_bar_chart(
            df=df_LP_provision,
            x_col=None,  # Use index as x-axis
            y_cols=provision_labels,
            title='5. Провизии по ссудному портфелю банков',
            height=600,
            width=800
        )

        return fig4, fig5
    except Exception as e:
        print(f"Error updating loan portfolio charts: {str(e)}")
        traceback.print_exc()
        # Return empty figures with error message
        empty_fig = go.Figure().add_annotation(
            text=f"Error loading data: {str(e)}",
            showarrow=False,
            font=dict(size=14, color="red")
        )
        return empty_fig, empty_fig

@app.callback(
    Output("graph-6", "figure"),
    Input("dd_graph-6", "value")
)
def update_interest_margin_chart(selected_item):
    """Update the interest margin chart based on the selected item."""
    try:
        # Check if data is available
        if df_IM.empty or selected_item is None:
            raise ValueError("Data or selected item not available")

        # Create traces
        data = []

        for bank in df_IM.index.unique():
            bank_data = df_IM[df_IM.index == bank]

            # Skip if no data for this bank
            if bank_data.empty or selected_item not in bank_data.columns:
                continue

            trace = go.Scatter(
                x=bank_data['Дата'],
                y=bank_data[selected_item],
                mode='markers+lines',
                name=bank,
                line=dict(width=3)
            )
            data.append(trace)

        layout = go.Layout(
            title={
                'text': '6. Динамика процентных доходов/расходов и маржи',
                'y': 0.9,
                'x': 0.5,
                'xanchor': 'center',
                'yanchor': 'top',
                'font': {'size': 18, 'color': 'black', 'family': "Arial Black, sans-serif"}
            },
            xaxis_title="Дата",
            yaxis_title=selected_item,
            template='plotly_white',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=-0.3,
                xanchor="center",
                x=0.5
            ),
            margin=dict(l=50, r=50, t=80, b=150),  # Extra bottom margin for legend
        )

        return go.Figure(data=data, layout=layout)
    except Exception as e:
        print(f"Error updating interest margin chart: {str(e)}")
        # Return empty figure with error message
        return go.Figure().add_annotation(
            text=f"Error loading data: {str(e)}",
            showarrow=False,
            font=dict(size=14, color="red")
        )

@app.callback(
    Output("graph-7", "figure"),
    Input("dd_graph-7", "value")
)
def update_prudential_norms_chart(selected_item):
    """Update the prudential norms chart based on the selected item."""
    try:
        # Check if data is available
        if df_PN.empty or selected_item is None:
            raise ValueError("Data or selected item not available")

        # Create traces
        data = []

        for bank in df_PN.index.unique():
            bank_data = df_PN[df_PN.index == bank]

            # Skip if no data for this bank
            if bank_data.empty or selected_item not in bank_data.columns:
                continue

            trace = go.Scatter(
                x=bank_data['Дата'],
                y=bank_data[selected_item],
                mode='markers+lines',
                name=bank,
                line=dict(width=3)
            )
            data.append(trace)

        layout = go.Layout(
            title={
                'text': '7. Динамика пруденциальных нормативов и их соблюдение',
                'y': 0.9,
                'x': 0.5,
                'xanchor': 'center',
                'yanchor': 'top',
                'font': {'size': 18, 'color': 'black', 'family': "Arial Black, sans-serif"}
            },
            xaxis_title="Дата",
            yaxis_title=selected_item,
            template='plotly_white',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=-0.3,
                xanchor="center",
                x=0.5
            ),
            margin=dict(l=50, r=50, t=80, b=150),  # Extra bottom margin for legend
        )

        fig = go.Figure(data=data, layout=layout)

        # Add threshold line if available
        try:
            if selected_item in df_PNT.index:
                threshold_value = df_PNT.loc[selected_item, 'T']

                # Add colored region based on threshold type
                if selected_item in df_floor_threshold.index:
                    # Floor threshold (should not be below this value)
                    fig.add_hrect(
                        y0=0,
                        y1=threshold_value,
                        line_width=0,
                        fillcolor="red",
                        opacity=0.2,
                        annotation_text="Min Threshold",
                        annotation_position="top right"
                    )
                    # Add threshold line
                    fig.add_hline(
                        y=threshold_value,
                        line_dash="dash",
                        line_color="red",
                        annotation_text=f"Min: {threshold_value}",
                        annotation_position="bottom right"
                    )
                else:
                    # Ceiling threshold (should not be above this value)
                    fig.add_hrect(
                        y0=threshold_value,
                        y1=threshold_value * 2,
                        line_width=0,
                        fillcolor="red",
                        opacity=0.2,
                        annotation_text="Max Threshold",
                        annotation_position="top right"
                    )
                    # Add threshold line
                    fig.add_hline(
                        y=threshold_value,
                        line_dash="dash",
                        line_color="red",
                        annotation_text=f"Max: {threshold_value}",
                        annotation_position="top right"
                    )
        except Exception as threshold_error:
            print(f"Error adding threshold: {str(threshold_error)}")
            # Continue without threshold rather than failing the entire graph

        return fig
    except Exception as e:
        print(f"Error updating prudential norms chart: {str(e)}")
        # Return empty figure with error message
        return go.Figure().add_annotation(
            text=f"Error loading data: {str(e)}",
            showarrow=False,
            font=dict(size=14, color="red")
        )

# Main entry point
if __name__ == "__main__":
    app.run(debug=False)

<IPython.core.display.Javascript object>